# Raw 데이터 수집

국가법령정보 API에서 부동산/임대차 관련 문서를 수집하여 `data/raw/{target}.jsonl`에 저장한다.

| target  | 설명                  | 중복 제거 기준       |
| ------- | --------------------- | -------------------- |
| `acr`   | 국민권익위원회 결정문 | `결정문일련번호`     |
| `eflaw` | 현행법령              | `법령ID`             |
| `prec`  | 판례                  | `판례일련번호`       |
| `expc`  | 법령해석례            | `법령해석례일련번호` |

> 각 섹션은 독립적으로 실행 가능. 전체 실행 시 **Kernel → Restart & Run All**.


## 0. 공통 설정


In [2]:
import sys

sys.path.insert(0, "..")
from src.api import fetch_list, save_raw, fetch_details

## 3. 판례 (prec)


In [ ]:
# 0. 임대차 분쟁에서 자주 다뤄지는 판례 관련 키워드
PREC_QUERIES: list[str] = [
    "주택",     # 주택: 거주와 직결된 임대차 판례들을 조회하기 위함
    "임대차",   # 임대차: 주제의 근간
    "계약"
]

# 1. 계약은 다른 두 단어와 달리 최고의 선택지는 아닌 것처럼 보입니다.
    # 하지만 임대차 계약 상에서의 시시비비를 다투는 판례를 찾는 것이 특약과 관련된 내용을 내포할 것이라 판단하여 계약을 넣었음.
# 2. 계약 대신 특약을 넣어도 됐을 거긴 한데, 차후 서비스가 확장될 가능성을 고려하면 계약이 더 포괄적이라 계약을 선택했음.
    # 필요에 따라 특약으로 넣어도 될 듯 합니다.

In [4]:
# 1. '판례 목록'에서 판례 키워드(PREC_QUERIES)를 내용에 포함하고 있는 판례들만 추출하기

seen_prec: set[str] = set() # 수집된 판례일련번호
prec_items: list[dict] = [] 

items = fetch_list("prec", query=PREC_QUERIES, max_items=None, extra_params={"search": 2})
    # 기존 코드에는 for문으로 했으나, query를 중복해서 주려면(n중쿼리) list를 넣어야해서 for문은 제거하고 query 리스트 전체 입력
for item in items:
    doc_id = str(item.get("판례일련번호", ""))
    # doc_id가 비어있거나 이미 수집한 항목이면 skip
    if doc_id and doc_id not in seen_prec:
        seen_prec.add(doc_id)
        prec_items.append(item)

save_raw(prec_items, target="prec", mode="w")

print(len(prec_items))

1717


In [5]:
# 2. 1번 과정에서 추출한 목록에 판례 내용을 추가하기.
prec_details = fetch_details(
  target="prec",
  items=prec_items,
  id_field="판례일련번호",
)

# 목록만 있는 prec.jsonl 덮어쓰기
result = save_raw(prec_details, target="prec", mode="w")
print(len(prec_details))

1717


---

## 5. 수집 결과 요약


In [2]:
from pathlib import Path

path = Path("../data/raw/prec.jsonl")

# JSONL은 한 줄 = 한 건이므로 줄 수 = 수집 건수
if path.exists():
    count = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"prec: {count:5d}건  →  {path}")
else:
    print(f"prec: 파일 없음")

prec:  1717건  →  ..\data\raw\prec.jsonl


---

---

---

# 수집한 RAW 데이터 전처리

- 전처리된 데이터는 data/processed 폴더에 넣어주세요

### 데이터 기본 정보

In [21]:
import json
import pandas as pd

df = pd.read_json(path, lines=True)   # lines = True : 한 줄에 JSON 하나

In [22]:
df

,id,사건번호,데이터출처명,사건종류코드,사건종류명,선고,선고일자,판례일련번호,판결유형,법원종류코드,법원명,판례상세링크,사건명,본문
0,1,2025두34604,대법원,400108,세무,선고,2026.01.29,616581,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,증여세부과처분취소,{'PrecService': {'판시사항': '<br/> [1] 구 상속세 및 증여...
1,2,2025가단107133,대법원,400101,민사,선고,2026.01.23,616251,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,배당이의,{'PrecService': {'판시사항': '<br/> 甲 소유의 주택에 관하여 ...
2,3,2025다213466,대법원,400101,민사,선고,2026.01.08,615767,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=615...,보증금,{'PrecService': {'판시사항': '<br/> [1] 주택임대차보호법 제...
3,4,수원고등법원-2025-나-12218,국세법령정보시스템,400101,민사,,2025.12.10,615921,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=615...,원고가 이 사건 경매절차에서 배당받을 권리가 있는지 여부,{'Law': '일치하는 판례가 없습니다. 판례명을 확인하여 주십시오.'}
4,5,2024다305087,대법원,400101,민사,선고,2025.12.04,613167,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=613...,공제금등청구의소[다세대주택 건물 중 임대의뢰인 소유의 특정 세대에 대한 임대차계약 ...,{'PrecService': {'판시사항': '<br/> [1] 개업공인중개사의 중...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1712,1713,,근로복지공단산재판례,400107,일반행정,,0001.01.01,404992,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=404...,산재보험료부과처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."
1713,1714,,근로복지공단산재판례,400107,일반행정,,0001.01.01,382020,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=382...,유족급여및장의비부지급처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."
1714,1715,,근로복지공단산재판례,400107,일반행정,,0001.01.01,408336,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=408...,유족급여및장의비부지급처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."
1715,1716,,근로복지공단산재판례,400107,일반행정,,0001.01.01,411292,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=411...,산재보험료부과처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."


---

### 메타데이터 / Content / 제외항목 전략 수립 및 실행

[ A. 메타데이터 ]
- 사건명
- 판례상세링크
- 판례정보일련번호: 유지보수 관점에서 필요 (판례가 수정된 경우 업데이트해야 함)
- 선고일자: 특정 기간을 지정해서 필터링해야할 수도 있다고 판단하였음.
- 법원명: 비어있는 값은 많으나, 필터링에 사용될 가능성이 높다고 판단하였음.
- 사건종류명
- ~~판결유형(본: 핵심 유형 등)~~
- 참조 조문
- 참조 판례

[ B. Content ]
- 판례 내용
- 판결 요지
- 판시사항

[ C. 제외항목 전략 수립 ]
- 사건번호: 우리는 하나의 사건에 대한 최종 법원만 가져올 것이기 때문
    - 전처리할 때, 한 사건에 대해 여러 건의 판례가 있는지 확인하는 용도로만 사용하고 제거
- 법원종류코드: 전체가 다 비었고, 법원명을 메타데이터로 올릴 것이라, 큰 메리트가 없음
- 판결유형: 약 1700건 중에 700건 가량이 판결유형 데이터가 존재하지 않음 + 포함해서 얻을 메리트가 크지 않아보임
- 선고: 억 800건 가량의 데이터가 존재하지 않음 + 포함해서 얻을 메리트가 크지 않아보임

In [23]:
df.info()       # 겉보기에는 멀쩡하지만 jsonl 윗부분만 보더라도 빈 문자열("")이 다수 존재함

<class 'pandas.DataFrame'>
RangeIndex: 1717 entries, 0 to 1716
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      1717 non-null   int64 
 1   사건번호    1717 non-null   str   
 2   데이터출처명  1717 non-null   str   
 3   사건종류코드  1717 non-null   int64 
 4   사건종류명   1717 non-null   str   
 5   선고      1717 non-null   str   
 6   선고일자    1717 non-null   str   
 7   판례일련번호  1717 non-null   int64 
 8   판결유형    1717 non-null   str   
 9   법원종류코드  1717 non-null   str   
 10  법원명     1717 non-null   str   
 11  판례상세링크  1717 non-null   str   
 12  사건명     1717 non-null   str   
 13  본문      1717 non-null   object
dtypes: int64(3), object(1), str(10)
memory usage: 187.9+ KB


In [24]:
# 각 컬럼별로 빈 문자열 개수
(df == "").sum()

### << Idea >>
# 1. 사건번호 같은 경우는 1700개의 비어있지 않은 문자열과 17개의 빈 문자열 값이 있음 
    # 엄밀하게 하고 싶다면 사건번호가 빈 건 온전하지 않으므로 그 판례들을 빼는 게 맞아보임
    # + 사건번호 중복 여부 파악 : 한 사건 당 하나의 (최종)판례만 존재한다면 중복이 없어야 함.
# 2. 법원명 / 판결유형 둘 모두 메타데이터로 활용하려 했으나 각각 약 40% 정도 되는 항목들이 비어 있음
    # (+) 법원명: 대법원의 판례를 보고 싶다 등의 질의는 충분히 나올 수 있는 반면에,
    # (-) 판결유형: 판결유형의 여부가 메타데이터 필터링이나 사용자 질의와 연관이 크게 없지 않을까 사료됨 (다만, 법조계 전문가가 본다면 다르게 생각할 수 있지 않을까라는 염려...)
# (-) 법원종류코드 :  다 비어있는, 무용지물
# 3. (-) 선고 : 선고도 '판결유형'과 동일한 이유로 제외하는 게 좋아보이나, 우선 어떤 데이터를 담고 있는지부터 확인해봐야 할 듯...

id           0
사건번호        17
데이터출처명       0
사건종류코드       0
사건종류명        0
선고         794
선고일자         0
판례일련번호       0
판결유형       728
법원종류코드    1717
법원명        726
판례상세링크       0
사건명          0
본문           0
dtype: int64

In [25]:
# 1. 사건 번호가 빈 케이스들 삭제
df = df[df["사건번호"] != ""]

# (df["사건번호"] == "").sum()
# df.info()

In [26]:
# 1. 중복된 사건 수는 없는가?
df["사건번호"].nunique()    # 중복 사건도 있네..

1687

In [27]:
# 1. 중복 사건 조회 >>> **조회만 하고 미조치**
df[df["사건번호"].duplicated(keep=False)]
    # 중복되는 모든 케이스가 사건번호가 같고, 법원명도 같은데, 판례일련번호만 다름... 본문을 대조해봐야 정확한 판단이 가능할 성 싶다.

,id,사건번호,데이터출처명,사건종류코드,사건종류명,선고,선고일자,판례일련번호,판결유형,법원종류코드,법원명,판례상세링크,사건명,본문
5,6,2024두67238,대법원,400108,세무,선고,2025.10.16,612881,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=612...,취득세등부과처분취소[직계존속으로부터 주택을 증여받으면서 임대차보증금반환채무를 인수한...,{'PrecService': {'판시사항': '<br/> [1] 조세법규의 해석 원...
7,8,2024두67238,지방세법령정보시스템,400107,일반행정,,2025.10.16,612679,파기환송,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=612...,취득세 등 부과 처분 취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."
45,46,2024가단106423,대법원,400101,민사,선고,2024.09.25,599623,판결 : 항소,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=599...,건물인도,{'PrecService': {'판시사항': ' 甲 지방공사가 공공주택 특별법 및 ...
46,47,2024가단106423,대법원,400101,민사,선고,2024.09.25,614407,판결 : 항소,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=614...,건물인도,{'PrecService': {'판시사항': '<br/> 甲 지방공사가 공공주택 특...
57,58,2023가단133077,대법원,400101,민사,선고,2024.08.09,241419,판결 : 항소,,대구지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=241...,손해배상(기),{'PrecService': {'판시사항': ' 공인중개사 甲이 임차인인 乙에게 ...
58,59,2023가단133077,대법원,400101,민사,선고,2024.08.09,614567,판결 : 항소,,대구지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=614...,손해배상(기),{'PrecService': {'판시사항': '<br/> 공인중개사 甲이 임차인인...
77,78,2023가단137284,대법원,400101,민사,선고,2024.05.23,241193,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=241...,부당이득금,{'PrecService': {'판시사항': ' 甲 주식회사 등이 집합건물의 구분...
79,80,2023가단137284,대법원,400101,민사,선고,2024.05.23,614575,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=614...,부당이득금,{'PrecService': {'판시사항': '<br/> 甲 주식회사 등이 집합건...
80,81,2023가단124967,대법원,400101,민사,선고,2024.05.21,614527,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=614...,보증금반환,{'PrecService': {'판시사항': '<br/> 甲과 임대차계약을 체결하...
81,82,2023가단124967,대법원,400101,민사,선고,2024.05.21,601889,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=601...,보증금반환,{'PrecService': {'판시사항': ' 甲과 임대차계약을 체결하고 甲 소...


In [28]:
# 2. '판결유형' 값 종류 확인
df["판결유형"].unique()     # 다양하긴 한데.. 해당 값이 법조계를 잘 아는 사람 아니면 큰 의미가 있는 컬럼은 아닐 것이라 생각됨.

<StringArray>
[             '판결',         '판결 : 확정',                '',            '파기환송',
          '처분청 승소',              '결정',          '처분청 패소',         '판결 : 항소',
              '기각',       '처분청 일부 패소',         '판결 : 상고',        '전원합의체 판결',
       '처분청 일부 승소',           '처분청패소',          '판결(주1)',           '처분청승소',
             '원고패',            '추가판결',        '결정 : 재항고',          '판결: 확정',
          '판결: 항소',             '원고승',          '판결: 상고',           '판결：상고',
           '판결：확정',           '판결：항소',       '판결 : 상고기각',         '판결：상고기각',
           '원고일부승',    '판결 : 항소기각·확정',         '판결 ： 확정',     '판결：항소심 조정성립',
    '판결 : 항소기각·상고',       '판결 ： 상고기각',         '판결 ： 항소',   '판결 ： 항소기각, 확정',
         '판결 ： 상고',    '제6민사부판결 : 확정',      '민사부판결 : 항소',    '제7민사부판결 : 확정',
    '제8특별부판결 : 확정', '제11민사부판결 : 상고기각',    '제3민사부판결 : 확정',    '제3민사부판결 : 상고',
    '제2민사부판결 : 확정',    '제4민사부판결 : 항소',   '제11민사부판결 : 확정',      '제1부판결 : 확정',
    '제1민사부판결 : 확정',      '가사부심판 : 항소',  '제1민사부판결 : 상고기각',     

In [29]:
# 3. '선고' 값 종류 확인
df["선고"].unique()

<StringArray>
['선고', '', '자', '선고,']
Length: 4, dtype: str

---

jsonl 내에서 다음과 같이 표기된 케이스들 제거: "본문": {"Law": "일치하는 판례가 없습니다.  판례명을 확인하여 주십시오."}

- 결론적으로는 이 과정을 제일 먼저 수행했으면 위의 과정이 많이 생략됐을 것 같음...

In [37]:
df["본문"].astype(str).str.contains("일치하는 판례가 없습니다").sum()   # 710개나 되네요

np.int64(710)

In [38]:
df = df[~df["본문"].astype(str).str.contains("일치하는 판례가 없습니다")]

df

,id,사건번호,데이터출처명,사건종류코드,사건종류명,선고,선고일자,판례일련번호,판결유형,법원종류코드,법원명,판례상세링크,사건명,본문
0,1,2025두34604,대법원,400108,세무,선고,2026.01.29,616581,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,증여세부과처분취소,{'PrecService': {'판시사항': '<br/> [1] 구 상속세 및 증여...
1,2,2025가단107133,대법원,400101,민사,선고,2026.01.23,616251,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,배당이의,{'PrecService': {'판시사항': '<br/> 甲 소유의 주택에 관하여 ...
2,3,2025다213466,대법원,400101,민사,선고,2026.01.08,615767,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=615...,보증금,{'PrecService': {'판시사항': '<br/> [1] 주택임대차보호법 제...
4,5,2024다305087,대법원,400101,민사,선고,2025.12.04,613167,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=613...,공제금등청구의소[다세대주택 건물 중 임대의뢰인 소유의 특정 세대에 대한 임대차계약 ...,{'PrecService': {'판시사항': '<br/> [1] 개업공인중개사의 중...
5,6,2024두67238,대법원,400108,세무,선고,2025.10.16,612881,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=612...,취득세등부과처분취소[직계존속으로부터 주택을 증여받으면서 임대차보증금반환채무를 인수한...,{'PrecService': {'판시사항': '<br/> [1] 조세법규의 해석 원...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,1696,4287행상82,대법원,400107,일반행정,선고,1954.03.11,86050,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=860...,행정처분취소,{'PrecService': {'판시사항': ' 적법한 자격없는 자의 불하신청과 귀...
1696,1697,4287민상216,대법원,400101,민사,선고,1954.03.10,86018,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=860...,부동산소유권이전등기,{'PrecService': {'판시사항': ' 양도담보 및 대물변제 예약의 효력<...
1697,1698,4287행3,대법원,400107,일반행정,선고,1954.02.27,232305,제2특별부판결: 확정,,서울고등법원,/DRF/lawService.do?OC=skn24&target=prec&ID=232...,행정처분취소청구사건,{'PrecService': {'판시사항': ' 귀속재산처리법상의 권리가 상속될 수...
1698,1699,4287행상76,대법원,400107,일반행정,선고,1954.02.04,86048,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=860...,행정처분취소,{'PrecService': {'판시사항': ' 행정처분 전의 소원 또는 진정서의 ...


In [40]:
(df == "").sum()

id          0
사건번호        0
데이터출처명      0
사건종류코드      0
사건종류명       0
선고         68
선고일자        0
판례일련번호      0
판결유형        2
법원종류코드    990
법원명         0
판례상세링크      0
사건명         0
본문          0
dtype: int64